# Airbnb Thessaloniki: Data Cleaning

**Source:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), Thessaloniki, Central Macedonia, Greece, data release of **24 September 2026**.

**Files:**
- `listings.csv`: one row per listing (detailed version, 90 columns)
- `calendar.csv`: availability (`t`/`f`) and minimum/maximum nights per listing per day; there is no price column
- `reviews.csv`: one row per guest review

The column definitions are in the Inside Airbnb data dictionary (linked on the *Get the Data* page).

## Introduction

*TODO (write at the end, Section 9): the goal of this notebook in one or two sentences and the main problems found in the raw data.*

### How this notebook is organised

*TODO (fill in at the end):*

1. Loading the data
2. First look: what is one row in each table?
3. Shape, dtypes and missing values
   - 3b. Empty and constant columns
4. Dtype conversion
5. Missing values by group
6. Keys, duplicates and referential integrity
7. Cross-table consistency
8. Value ranges, typos and date order
9. Export and summary

Imports, and show all columns: `listings` has 90 and pandas would hide most of them.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Loading the data

Load the three raw CSV files. These DataFrames are never changed; every fix will be done on a `_clean` copy.

In [2]:
listings = pd.read_csv("data/listings.csv")
calendar = pd.read_csv("data/calendar.csv")
reviews = pd.read_csv("data/reviews.csv")

Put the three tables in a dict, so every check can run in one loop instead of three identical cells.

In [3]:
dataframes = {
    "listings": listings,
    "calendar": calendar,
    "reviews": reviews
}

## 2. First look: what is one row in each table?

**Hypothesis:** `listings` has one row per listing, `calendar` has one row per listing per day for the next year (so about 365 rows per listing), and `reviews` has one row per guest review.

Size of each table (rows, columns).

In [4]:
for name, df in dataframes.items():
    print(f"{name} shape: {df.shape}")

listings shape: (4961, 90)
calendar shape: (1810765, 5)
reviews shape: (244369, 6)


Calendar rows per listing: tests the hypothesis that each listing has one row per day.

In [5]:
calendar.shape[0] / listings.shape[0]

365.0

Distinct dates in `calendar`: if every listing covered the same year, there would be 365.

In [6]:
calendar["date"].nunique()

366

**Findings:**
- `listings` has 4,961 rows and 90 columns, `calendar` has 1,810,765 rows and `reviews` has 244,369 rows.
- `calendar` rows / `listings` rows = 365.0. This means every listing has exactly one year of daily rows, which supports the hypothesis.
- But `calendar.date` has 366 distinct dates, so not all listings start on the same day. This is checked in Section 6.

### Grain of each table

| table | one row is | candidate key | status |
|---|---|---|---|
| `listings` | one Airbnb listing | `id` | verified |
| `calendar` | one listing on one date | (`listing_id`, `date`) | verified |
| `reviews` | one guest review | `id` | verified |

**Relationships:** `calendar.listing_id` and `reviews.listing_id` both point to `listings.id`. The status was checked in Section 6 (6a and 6f).

## 3. Shape, dtypes and missing values

`info()` gives a quick overview of each table: number of rows, the dtype of every column and how many values are not missing.

In [7]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 4961 entries, 0 to 4960
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            4961 non-null   int64  
 1   listing_url                                   4961 non-null   str    
 2   scrape_id                                     4961 non-null   int64  
 3   last_scraped                                  4961 non-null   str    
 4   source                                        435 non-null    str    
 5   name                                          4961 non-null   str    
 6   description                                   4960 non-null   str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   4961 non-null   str    
 9   host_id                                       4961 non-null   int64  
 10 

The same overview for `calendar`.

In [8]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 1810765 entries, 0 to 1810764
Data columns (total 5 columns):
 #   Column          Dtype
---  ------          -----
 0   listing_id      int64
 1   date            str  
 2   available       str  
 3   minimum_nights  int64
 4   maximum_nights  int64
dtypes: int64(3), str(2)
memory usage: 69.1 MB


The same overview for `reviews`.

In [9]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 244369 entries, 0 to 244368
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   listing_id     244369 non-null  int64
 1   id             244369 non-null  int64
 2   date           244369 non-null  str  
 3   reviewer_id    244369 non-null  int64
 4   reviewer_name  244369 non-null  str  
 5   comments       244355 non-null  str  
dtypes: int64(3), str(3)
memory usage: 11.2 MB


`info()` is hard to read with 90 columns, so `column_summary` builds one row per column with its dtype, the number and percentage of NaN, and the number of distinct values. The most empty columns come first.

In [10]:
def column_summary(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "n_nan": df.isna().sum(),
        "pct_nan": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique()
    })
    return summary.sort_values("n_nan", ascending=False)

for name, df in dataframes.items():
    print(name)
    display(column_summary(df))

listings


,dtype,n_nan,pct_nan,n_unique
neighborhood_overview,float64,4961,100.0,0
host_since,float64,4961,100.0,0
host_response_time,float64,4961,100.0,0
host_thumbnail_url,float64,4961,100.0,0
host_acceptance_rate,float64,4961,100.0,0
...,...,...,...,...
availability_30,int64,0,0.0,31
calculated_host_listings_count,int64,0,0.0,35
calculated_host_listings_count_entire_homes,int64,0,0.0,37
calculated_host_listings_count_private_rooms,int64,0,0.0,9


calendar


,dtype,n_nan,pct_nan,n_unique
listing_id,int64,0,0.0,4961
date,str,0,0.0,366
available,str,0,0.0,2
minimum_nights,int64,0,0.0,36
maximum_nights,int64,0,0.0,131


reviews


,dtype,n_nan,pct_nan,n_unique
comments,str,14,0.0,227596
listing_id,int64,0,0.0,4390
id,int64,0,0.0,244369
date,str,0,0.0,4525
reviewer_id,int64,0,0.0,194620
reviewer_name,str,0,0.0,34248


Keep only the `listings` columns that have at least one missing value.

In [11]:
s = column_summary(listings)
s[s["n_nan"] > 0]

,dtype,n_nan,pct_nan,n_unique
neighborhood_overview,float64,4961,100.0,0
host_since,float64,4961,100.0,0
host_response_time,float64,4961,100.0,0
host_thumbnail_url,float64,4961,100.0,0
host_acceptance_rate,float64,4961,100.0,0
host_response_rate,float64,4961,100.0,0
host_verifications,float64,4961,100.0,0
neighbourhood,float64,4961,100.0,0
host_total_listings_count,float64,4961,100.0,0
host_neighbourhood,float64,4961,100.0,0


The `str` columns of `listings` with their first three values, to see which ones are really text and which ones hide dates, flags, numbers or lists.

In [12]:
obj_cols = s[s["dtype"] == "str"].index
listings[obj_cols].head(3).T

,0,1,2
source,NaN,NaN,NaN
host_about,NaN,Με όραμα την Ελληνική Φιλοξενία και μοναδικό σ...,NaN
host_location,NaN,"Thessaloniki, Greece",NaN
last_review,2026-09-06,2026-08-20,NaN
first_review,2026-03-15,2026-03-08,NaN
price,$96.60,$40.00,$60.00
price_quote_raw,"{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...","{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...","{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""..."
price_quote_checkin_date,2026-09-24,2026-09-30,2026-10-03
price_quote_checkout_date,2026-09-26,2026-10-01,2026-10-05
license,00003427019,3672670,00003673975


Which currency each price quote names, read from the JSON text in `price_quote_raw`.

In [13]:
# currency named inside each price quote (JSON text)
currency = listings["price_quote_raw"].str.extract(r'"currency":\s*"(\w+)"')[0]
currency.value_counts(dropna=False)

0
EUR    4632
NaN     329
Name: count, dtype: int64

**Findings:** the `str` columns fall into five groups.
- **Dates stored as text** (`YYYY-MM-DD`): `last_scraped`, `calendar_last_scraped`, `first_review`, `last_review`, `price_quote_checkin_date`, `price_quote_checkout_date`. Converted in Section 4a.
- **Yes/no flags stored as `t`/`f`**: `host_is_superhost`, `host_identity_verified`, `host_has_profile_pic`, `has_availability`.
- **A number hidden in text**: `price` (`$96.60`) and `bathrooms_text` (`1 bath`, `1.5 baths`, `Half-bath`).
- **A list or JSON stored as text**: `amenities` (a list of amenities) and `price_quote_raw` (the full price quote as JSON).
- **Real text** that stays as it is: names, descriptions, URLs and categories (`name`, `description`, `host_name`, `host_about`, `host_location`, `room_type`, `property_type`, `neighbourhood_cleansed`, the `*_url` columns, `source`).

Two things to keep in mind:
- `price` is written with a `$` sign, but `price_quote_raw` says `"currency": "EUR"`. Every quote that names a currency says EUR (checked above: 4,632 quotes say EUR, 329 name no currency), so the `$` is only a display symbol and the prices are in **euros**.
- `license` is a registration number, so it stays text. Its values are not written the same way (`00003427019` with leading zeros, `3672670` without). This is looked at in Section 5.

`float64` columns with no values at all. pandas gives `float64` to a completely empty column, so here the dtype says nothing about the content. These columns are handled in Section 3b.

In [14]:
s[(s["dtype"] == "float64") & (s["pct_nan"] == 100)]

,dtype,n_nan,pct_nan,n_unique
neighborhood_overview,float64,4961,100.0,0
host_since,float64,4961,100.0,0
host_response_time,float64,4961,100.0,0
host_thumbnail_url,float64,4961,100.0,0
host_acceptance_rate,float64,4961,100.0,0
host_response_rate,float64,4961,100.0,0
host_verifications,float64,4961,100.0,0
neighbourhood,float64,4961,100.0,0
host_total_listings_count,float64,4961,100.0,0
host_neighbourhood,float64,4961,100.0,0


Columns with only one distinct value. `nunique()` ignores NaN, so a column with one value plus NaN also shows up here. Whether each one is truly constant is checked in Section 3b.

In [15]:
s[s["n_unique"] == 1]

,dtype,n_nan,pct_nan,n_unique
source,str,4526,91.2,1
has_availability,str,11,0.2,1
scrape_id,int64,0,0.0,1


The id columns: how many digits they have, and whether they would survive a conversion to `float64`.

In [16]:
for name, df in dataframes.items():
    for col in df.columns:
        if col.endswith("id"):
            biggest = df[col].max()
            print(name, col, df[col].dtype, len(str(biggest)), "digits,",
                  "safe as float:", int(float(biggest)) == biggest , "\n")

listings id int64 19 digits, safe as float: False 

listings scrape_id int64 14 digits, safe as float: True 

listings host_id int64 19 digits, safe as float: False 

listings host_profile_id int64 19 digits, safe as float: False 

calendar listing_id int64 19 digits, safe as float: False 

reviews listing_id int64 19 digits, safe as float: False 

reviews id int64 19 digits, safe as float: False 

reviews reviewer_id int64 19 digits, safe as float: False 



For every id column: how many values are above 2⁵³ and how many change after a round trip through float.

In [17]:
# how many ids are above 2**53, and how many change after a round trip through float
for name, df in dataframes.items():
    for col in df.columns:
        if col.endswith("id"):
            digits = df[col].astype(str).str.len()
            n_big = (df[col] > 2**53).sum()
            n_changed = (df[col].astype(float).astype("int64") != df[col]).sum()
            print(f"{name}.{col}: {digits.min()}-{digits.max()} digits, "
                  f"{n_big:,} values > 2**53, {n_changed:,} change through float")

listings.id: 6-19 digits, 3,812 values > 2**53, 3,796 change through float
listings.scrape_id: 14-14 digits, 0 values > 2**53, 0 change through float
listings.host_id: 6-19 digits, 119 values > 2**53, 119 change through float
listings.host_profile_id: 19-19 digits, 4,961 values > 2**53, 4,952 change through float
calendar.listing_id: 6-19 digits, 1,391,380 values > 2**53, 1,385,540 change through float
reviews.listing_id: 6-19 digits, 100,278 values > 2**53, 99,696 change through float
reviews.id: 7-19 digits, 194,652 values > 2**53, 193,421 change through float
reviews.reviewer_id: 4-19 digits, 2,266 values > 2**53, 2,263 change through float


**Findings:** the `id` in `listings` has between 6 and 19 digits. 3,812 of the 4,961 ids are larger than 2⁵³, the largest integer float64 stores exactly, and a round trip through float changes 3,796 of them. The same problem appears in `host_id`, `host_profile_id`, `calendar.listing_id`, `reviews.listing_id`, `reviews.id` and `reviews.reviewer_id`; only `scrape_id` (14 digits) is safe.

int64 holds them correctly, so the columns stay int64. But any step that introduces a NaN into an id column (e.g. a merge) would silently turn it into float and corrupt these ids, so I check the id dtypes after every merge.

### Results

- **`listings`**:
  - Missing values: 14 columns are 100% empty. `source` is 91% NaN, `host_about` 55% and `host_location` 37%. The review columns (`first_review`, `last_review`, `reviews_per_month`, `review_scores_*`) are all missing in 571 rows (11.5%), and the price columns (`price`, `price_quote_*`, `estimated_revenue_l365d`) in 300–334 rows (6.0–6.7%). Smaller gaps: `license` (105), `beds` (48), `has_availability` (11), the `*_nights` columns (3), `bathrooms_text` and `description` (1 each).
  - Wrong dtypes: the dates (`last_scraped`, `calendar_last_scraped`, `first_review`, `last_review`, `price_quote_checkin_date`, `price_quote_checkout_date`) are text. The t/f columns (`host_is_superhost`, `host_has_profile_pic`, `host_identity_verified`, `has_availability`) are text. `price` is text with a `$` sign. `bathrooms_text` and `amenities` hide a number and a list inside text. `minimum_nights` and `maximum_nights` are float only because of 3 NaN.
- **`calendar`**: no missing values. `date` and `available` (t/f) are text.
- **`reviews`**: 14 reviews have no `comments`. `date` is text.
- **All tables**: the id columns are int64 with up to 19 digits and would be corrupted if they became float.

Nothing is fixed yet. This list is the work for Sections 3b–5.

Make a `_clean` copy of each table and put the copies in a dict; every fix from here on goes on these copies, so the raw tables stay unchanged.

In [18]:
listings_clean = listings.copy()
calendar_clean = calendar.copy()
reviews_clean = reviews.copy()

dataframes_clean = {
    "listings": listings_clean, "calendar": calendar_clean, "reviews": reviews_clean
}



### 3b. Empty and constant columns

**Hypothesis:** the 14 columns of `listings` that are 100% empty hold no information that is not in another column, so they can be dropped. Of the 3 columns with one distinct value, only the ones with no NaN are truly constant.

Find the `listings` columns that are 100% empty (NaN in every row).

In [19]:
empty_cols = listings_clean.columns[listings_clean.isna().all()]
print(len(empty_cols))
list(empty_cols)

14


['neighborhood_overview',
 'host_since',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_thumbnail_url',
 'host_neighbourhood',
 'host_total_listings_count',
 'host_verifications',
 'neighbourhood',
 'neighbourhood_group_cleansed',
 'bathrooms',
 'calendar_updated',
 'instant_bookable']

Before dropping an empty column, check whether the same information exists in another column:

| empty column | related column | has values? | decision |
|---|---|---|---|
| `bathrooms` | `bathrooms_text` | yes (1 NaN) | drop; the number can be extracted from `bathrooms_text` in Section 4 |
| `host_since` | `hosts_time_as_host_years` / `_months` | yes (0 NaN) | drop; the time as host is still known in years and months |
| `neighbourhood` | `neighbourhood_cleansed` | yes (0 NaN, 7 areas) | drop; `neighbourhood_cleansed` is used instead |


In [20]:
listings_clean[["bathrooms", "bathrooms_text"]].head(10)


,bathrooms,bathrooms_text
0,NaN,1 bath
1,NaN,1 bath
2,NaN,1 bath
3,NaN,1 bath
4,NaN,1 bath
5,NaN,1 bath
6,NaN,1 bath
7,NaN,1 bath
8,NaN,1 bath
9,NaN,1 bath


Missing values in `bathrooms` and `bathrooms_text`.

In [21]:
listings_clean[["bathrooms", "bathrooms_text"]].isna().sum()


bathrooms         4961
bathrooms_text       1
dtype: int64

The most common values of `bathrooms_text`, to see which number is written in the text.

In [22]:
listings_clean["bathrooms_text"].value_counts().head(10)


bathrooms_text
1 bath            4325
1.5 baths          251
2 baths            199
1 private bath     114
1 shared bath       17
3 baths             11
0 baths             11
2.5 baths           11
Half-bath            8
4 baths              4
Name: count, dtype: int64

Missing values in `neighbourhood` and `neighbourhood_cleansed`, and how many listings are in each area.

In [23]:
print(listings_clean[["neighbourhood", "neighbourhood_cleansed"]].isna().sum())
listings_clean["neighbourhood_cleansed"].value_counts()


neighbourhood             4961
neighbourhood_cleansed       0
dtype: int64


neighbourhood_cleansed
Thessaloniki              4404
Kalamaria                  177
Neapolis – Sykeon          125
Pilea – Chortiatis          74
Pavlou Mela                 68
Ampelokipon – Menemeni      65
Kordelio – Evosmos          48
Name: count, dtype: int64

`host_since` next to the two columns that also give the time as host: how many NaN each has, and the first rows.

In [24]:
host_cols = ["host_since", "hosts_time_as_host_years", "hosts_time_as_host_months"]
print(listings_clean[host_cols].isna().sum())
listings_clean[host_cols].head()

host_since                   4961
hosts_time_as_host_years        0
hosts_time_as_host_months       0
dtype: int64


,host_since,hosts_time_as_host_years,hosts_time_as_host_months
0,NaN,0,11
1,NaN,7,1
2,NaN,0,4
3,NaN,9,4
4,NaN,9,4


**Check:** `host_since` is empty in all rows, but `hosts_time_as_host_years` and `_months` have no NaN, so the time as host is still known. `neighbourhood` is empty in all rows, while `neighbourhood_cleansed` has no NaN and 7 areas. Thessaloniki holds 4,404 of the 4,961 listings (89%), and the other 6 areas have between 48 and 177 each. Both empty columns can be dropped without losing information.


The single-value columns found in Section 3 are checked again with `nunique(dropna=False)`, because `nunique()` ignores NaN.

In [25]:
single_cols = s[s["n_unique"] == 1].index
listings_clean[single_cols].nunique(dropna=False)


source              2
has_availability    2
scrape_id           1
dtype: int64

The values of `source` and `has_availability`, with NaN counted.

In [26]:
for col in ["source", "has_availability"]:
    print(listings_clean[col].value_counts(dropna=False), "\n")


source
NaN                4526
previous scrape     435
Name: count, dtype: int64 

has_availability
t      4950
NaN      11
Name: count, dtype: int64 



**Findings:**
- Truly constant (to drop): `scrape_id`, the same value in all 4,961 rows.
- One value + NaN (kept for now, acts as a yes/no flag): `source` ("previous scrape" in 435 rows, NaN in 4,526) and `has_availability` ("t" in 4,950 rows, NaN in 11).

Shape of `listings_clean` before the drop.

In [27]:
listings_clean.shape

(4961, 90)

Drop the 14 empty columns and `scrape_id`, and print the shape after.

In [28]:
cols_to_drop = list(empty_cols) + ["scrape_id"]
listings_clean = listings_clean.drop(columns=cols_to_drop)
dataframes_clean["listings"] = listings_clean
print(len(cols_to_drop), "columns dropped")
listings_clean.shape


15 columns dropped


(4961, 75)

**Dropped columns (15):**
- 14 columns that are 100% empty: `neighborhood_overview`, `host_since`, `host_response_time`, `host_response_rate`, `host_acceptance_rate`, `host_thumbnail_url`, `host_neighbourhood`, `host_total_listings_count`, `host_verifications`, `neighbourhood`, `neighbourhood_group_cleansed`, `bathrooms`, `calendar_updated`, `instant_bookable`. None of them holds information that is not in another column: the number of bathrooms is in `bathrooms_text`, the time as host is in `hosts_time_as_host_years` / `_months`, and the area is in `neighbourhood_cleansed`.
- `scrape_id`: the same value in all 4,961 rows.

**Kept:** `source` and `has_availability` have one value plus NaN, so the NaN separates the listings into two groups. Their meaning is checked in Section 5.

`listings_clean` now has 4,961 rows and 75 columns. No rows were dropped.


## 4. Dtype conversion

### 4a. Dates

**Hypothesis:** all date columns are text in the format `YYYY-MM-DD`, so `pd.to_datetime` converts them without creating new NaN. The NaN count before and after the conversion should be the same.


Check the format first: a `YYYY-MM-DD` string always has 10 characters, so count the string lengths in each date column of `listings_clean`.

In [29]:
date_cols = ["last_scraped", "calendar_last_scraped", "first_review", "last_review",
             "price_quote_checkin_date", "price_quote_checkout_date"]
for col in date_cols:
    print(col, listings_clean[col].dropna().str.len().value_counts().to_dict())

last_scraped {10: 4961}
calendar_last_scraped {10: 4961}
first_review {10: 4390}
last_review {10: 4390}
price_quote_checkin_date {10: 4661}
price_quote_checkout_date {10: 4661}


Number of NaN in the date columns before the conversion.

In [30]:
nan_before = listings_clean[date_cols].isna().sum()
nan_before

last_scraped                   0
calendar_last_scraped          0
first_review                 571
last_review                  571
price_quote_checkin_date     300
price_quote_checkout_date    300
dtype: int64

Convert with an explicit `format`, so that a value written in another format raises an error instead of silently becoming NaN.

In [31]:
for col in date_cols:
    listings_clean[col] = pd.to_datetime(listings_clean[col], format="%Y-%m-%d")

The dtype and the NaN after the conversion, next to the NaN before.

In [32]:
pd.DataFrame({
    "dtype": listings_clean[date_cols].dtypes,
    "nan_before": nan_before,
    "nan_after": listings_clean[date_cols].isna().sum()
})

,dtype,nan_before,nan_after
last_scraped,datetime64[us],0,0
calendar_last_scraped,datetime64[us],0,0
first_review,datetime64[us],571,571
last_review,datetime64[us],571,571
price_quote_checkin_date,datetime64[us],300,300
price_quote_checkout_date,datetime64[us],300,300


The same check and conversion for the `date` column of `calendar` and `reviews`.

In [33]:
for name in ["calendar", "reviews"]:
    df = dataframes_clean[name]
    print(name, "string lengths:", df["date"].str.len().value_counts().to_dict())
    n_before = df["date"].isna().sum()
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
    print(name, df["date"].dtype, "NaN before:", n_before, "NaN after:", df["date"].isna().sum())

calendar string lengths: {10: 1810765}
calendar datetime64[us] NaN before: 0 NaN after: 0
reviews string lengths: {10: 244369}
reviews datetime64[us] NaN before: 0 NaN after: 0


The first and the last date of `calendar` and `reviews`.

In [34]:
for name in ["calendar", "reviews"]:
    df = dataframes_clean[name]
    print(name, df["date"].min().date(), "to", df["date"].max().date(), "|", df["date"].nunique(), "distinct dates")

calendar 2026-09-23 to 2027-09-23 | 366 distinct dates
reviews 2012-06-02 to 2026-09-24 | 4525 distinct dates


**Findings:** the hypothesis holds.
- Every non-missing value in the 8 date columns has 10 characters, and `pd.to_datetime` with `format="%Y-%m-%d"` raised no error, so they are all written as `YYYY-MM-DD`.
- The conversion created no new NaN: `first_review` and `last_review` keep 571, `price_quote_checkin_date` and `price_quote_checkout_date` keep 300, and `last_scraped`, `calendar_last_scraped`, `calendar.date` and `reviews.date` have 0.
- All 8 columns are now `datetime64[us]`.
- `calendar` runs from 2026-09-23 to 2027-09-23 (366 distinct dates) and `reviews` from 2012-06-02 to 2026-09-24 (4,525 distinct dates). Why `calendar` has 366 dates while each listing has 365 rows on average is checked in Section 6.

### 4b. Booleans

**Hypothesis:** the five flag columns hold only `t`, `f` and NaN, so they convert to the nullable `boolean` dtype without new NaN, and `has_availability` keeps its 11 NaN.

The values of the flag columns before the conversion, with NaN counted.

In [35]:
bool_cols = ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "has_availability"]
for col in bool_cols:
    print(col, listings_clean[col].value_counts(dropna=False).to_dict())
print("available", calendar_clean["available"].value_counts(dropna=False).to_dict())

host_is_superhost {'f': 3081, 't': 1880}
host_has_profile_pic {'t': 4694, 'f': 267}
host_identity_verified {'t': 4922, 'f': 39}
has_availability {'t': 4950, nan: 11}
available {'t': 1151417, 'f': 659348}


Map `t` to True and `f` to False, then use `boolean`, which can hold NaN. `astype(bool)` on the text would be wrong: every non-empty string, even `"f"`, becomes True.

In [36]:
flag_map = {"t": True, "f": False}
for col in bool_cols:
    listings_clean[col] = listings_clean[col].map(flag_map).astype("boolean")
calendar_clean["available"] = calendar_clean["available"].map(flag_map).astype("boolean")

The same counts after the conversion, with the new dtype.

In [37]:
for col in bool_cols:
    print(col, listings_clean[col].dtype, listings_clean[col].value_counts(dropna=False).to_dict())
print("available", calendar_clean["available"].dtype, calendar_clean["available"].value_counts(dropna=False).to_dict())

host_is_superhost boolean {np.False_: 3081, np.True_: 1880}
host_has_profile_pic boolean {np.True_: 4694, np.False_: 267}
host_identity_verified boolean {np.True_: 4922, np.False_: 39}
has_availability boolean {np.True_: 4950, <NA>: 11}
available boolean {np.True_: 1151417, np.False_: 659348}


**Findings:** the hypothesis holds. The columns held only `t`, `f` and NaN, and all five are now `boolean` with the same counts:

| column | True | False | NaN |
|---|---|---|---|
| `host_is_superhost` | 1,880 | 3,081 | 0 |
| `host_has_profile_pic` | 4,694 | 267 | 0 |
| `host_identity_verified` | 4,922 | 39 | 0 |
| `has_availability` | 4,950 | 0 | 11 |
| `calendar.available` | 1,151,417 | 659,348 | 0 |

`has_availability` kept its 11 NaN and has no False at all. What those NaN mean is looked at in Section 5.

### 4c. Price

**Hypothesis:** `price` is text like `$1,651.00`, so removing the `$` and the thousands commas gives a float with no new NaN, and the result equals `price_quote_price_per_night` on every row. Prices are in euros; the `$` is only a display symbol.

Check the format before changing anything: the number of NaN, how many values do not look like `$1,651.00`, and the values with a thousands comma.

In [38]:
price_nan_before = listings_clean["price"].isna().sum()
not_matching = ~listings_clean["price"].dropna().str.fullmatch(r"\$[\d,]+\.\d\d")
print("NaN:", price_nan_before, "| not in the expected format:", not_matching.sum())
listings_clean.loc[listings_clean["price"].str.contains(",", regex=False, na=False), "price"]

NaN: 334 | not in the expected format: 0


135      $1,651.00
790     $12,021.50
1477     $5,776.50
2979     $2,316.27
4242    $10,700.50
4294     $1,157.00
Name: price, dtype: str

Remove the `$` and the commas as plain text (`regex=False`, because `$` means "end of string" in a regex), then convert to float.

In [39]:
listings_clean["price"] = (
    listings_clean["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)
print(listings_clean["price"].dtype, "| NaN before:", price_nan_before, "| NaN after:", listings_clean["price"].isna().sum())

float64 | NaN before: 334 | NaN after: 334


Compare with `price_quote_price_per_night`: rows where only one of the two is NaN, and rows where both have a value but they differ.

In [40]:
price = listings_clean["price"]
quote = listings_clean["price_quote_price_per_night"]
both = price.notna() & quote.notna()
print("rows with both values:", both.sum())
print("NaN in only one of the two:", (price.isna() != quote.isna()).sum())
print("rows that differ:", (price[both] != quote[both]).sum())
print("min:", price.min(), "| max:", price.max())

rows with both values: 4627
NaN in only one of the two: 0
rows that differ: 0
min: 18.29 | max: 12021.5


**Findings:** the hypothesis holds.
- Every non-missing value had the expected format (0 did not), and 6 prices had a thousands comma, for example `$1,651.00` and `$12,021.50`.
- `price` is now `float64`, in euros, with no new NaN: 334 before and 334 after.
- `price` equals `price_quote_price_per_night` on all 4,627 rows that have a value, and no row has a NaN in only one of the two. The two columns hold the same information; both stay for now.
- The prices run from 18.29 to 12,021.50. The very high values were not touched: outliers are checked in Section 8.

### 4d. Bathrooms

**Hypothesis:** every value of `bathrooms_text` starts with a number, except the half-baths, which are written in words. The word `shared` marks shared bathrooms. So the text splits into a float `bathrooms` and a boolean `bath_shared`, with NaN only where the text itself is missing.

All the values of `bathrooms_text`, with NaN counted.

In [41]:
listings_clean["bathrooms_text"].value_counts(dropna=False)

bathrooms_text
1 bath               4325
1.5 baths             251
2 baths               199
1 private bath        114
1 shared bath          17
3 baths                11
0 baths                11
2.5 baths              11
Half-bath               8
4 baths                 4
2 shared baths          3
0 shared baths          2
Private half-bath       2
1.5 shared baths        1
5 baths                 1
NaN                     1
Name: count, dtype: int64

Take the number at the start of the text. A half-bath (toilet and sink, but no shower or bathtub) has no number, so I set it to 0.5, which is how the text itself reads: half of one bathroom.

In [42]:
bath_text = listings_clean["bathrooms_text"]
bath_number = bath_text.str.extract(r"^(\d+(?:\.\d+)?)")[0].astype(float)
is_half = bath_text.str.contains("half", case=False, na=False)
listings_clean["bathrooms"] = bath_number.mask(is_half, 0.5)

`bath_shared` is True when the text says `shared`. `na=False` would turn the missing text into False, so that row is set back to NA: we do not know if it is shared.

In [43]:
is_shared = bath_text.str.contains("shared", case=False, na=False).astype("boolean")
listings_clean["bath_shared"] = is_shared.mask(bath_text.isna(), pd.NA)

Each text value next to the two new columns, to see that every one was read correctly. `bathrooms_text` is kept.

In [44]:
listings_clean[["bathrooms_text", "bathrooms", "bath_shared"]].value_counts(dropna=False)

bathrooms_text     bathrooms  bath_shared
1 bath             1.0        False          4325
1.5 baths          1.5        False           251
2 baths            2.0        False           199
1 private bath     1.0        False           114
1 shared bath      1.0        True             17
3 baths            3.0        False            11
0 baths            0.0        False            11
2.5 baths          2.5        False            11
Half-bath          0.5        False             8
4 baths            4.0        False             4
2 shared baths     2.0        True              3
0 shared baths     0.0        True              2
Private half-bath  0.5        False             2
1.5 shared baths   1.5        True              1
5 baths            5.0        False             1
NaN                NaN        <NA>              1
Name: count, dtype: int64

The dtype and the NaN of the two new columns.

In [45]:
print(listings_clean[["bathrooms_text", "bathrooms", "bath_shared"]].dtypes)
listings_clean[["bathrooms_text", "bathrooms", "bath_shared"]].isna().sum()

bathrooms_text        str
bathrooms         float64
bath_shared       boolean
dtype: object


bathrooms_text    1
bathrooms         1
bath_shared       1
dtype: int64

**Findings:** the hypothesis holds. `bathrooms_text` has 15 distinct values plus NaN, and each one maps to the expected pair in the table above.
- `bathrooms` is `float64`. The 10 half-baths (`Half-bath` 8, `Private half-bath` 2) were set to 0.5, because they have no number in the text.
- `bath_shared` is `boolean` and is True for 23 listings (`1 shared bath` 17, `2 shared baths` 3, `0 shared baths` 2, `1.5 shared baths` 1). `1 private bath` counts as not shared.
- No new NaN: the 1 listing with no `bathrooms_text` is also missing in both new columns. `bathrooms_text` is kept.
- 13 listings have 0 bathrooms (`0 baths` 11, `0 shared baths` 2). They are kept as written and belong to the range checks of Section 8.

### 4e. Amenities

**Hypothesis:** `amenities` is a JSON list of strings stored as text, so the number of amenities is the number of `", "` separators plus one.

Check what could break the count: missing values, empty lists, and escaped quotes inside an amenity name.

In [46]:
print("NaN:", listings_clean["amenities"].isna().sum())
print("empty lists:", (listings_clean["amenities"] == "[]").sum())
print("escaped quotes:", listings_clean["amenities"].str.contains('\\"', regex=False).sum())
print(listings_clean["amenities"].iloc[0])

NaN: 0
empty lists: 0
escaped quotes: 0
["Air conditioning", "Smoking allowed", "Kitchen", "Outdoor dining area", "Dedicated workspace"]


Count the separators and add one. `amenities` is kept.

In [47]:
listings_clean["n_amenities"] = (listings_clean["amenities"].str.count('", "') + 1).astype("int64")

A second way to count as a check: every amenity has two quotes around it, so the number of quotes divided by 2 must give the same result.

In [48]:
n_from_quotes = listings_clean["amenities"].str.count('"') / 2
print("rows that differ:", (n_from_quotes != listings_clean["n_amenities"]).sum())
print(listings_clean["n_amenities"].dtype)
listings_clean["n_amenities"].describe()

rows that differ: 0
int64


count    4961.000000
mean       33.416448
std        13.065002
min         1.000000
25%        26.000000
50%        35.000000
75%        42.000000
max        87.000000
Name: n_amenities, dtype: float64

**Findings:** the hypothesis holds.
- `amenities` has 0 NaN, 0 empty lists and 0 escaped quotes, so counting the separators is safe.
- `n_amenities` is `int64`. The second count, from the quotes, differs on 0 rows.
- Listings have between 1 and 87 amenities, with a median of 35. `amenities` is kept.

### 4f. Nights

**Hypothesis:** the 6 `*_nights` columns are float only because of 3 NaN and have no decimals, so they convert to the nullable `Int64` without losing anything. Plain `int64` cannot hold NaN.

For each column, the number of NaN and of values with decimals, and how many rows have a NaN in any and in all of the columns.

In [49]:
night_cols = ["minimum_nights", "maximum_nights", "minimum_minimum_nights",
              "maximum_minimum_nights", "minimum_maximum_nights", "maximum_maximum_nights"]
for col in night_cols:
    values = listings_clean[col]
    print(col, values.dtype, "| NaN:", values.isna().sum(), "| with decimals:", (values.dropna() % 1 != 0).sum())
print("rows with NaN in any:", listings_clean[night_cols].isna().any(axis=1).sum(),
      "| in all:", listings_clean[night_cols].isna().all(axis=1).sum())

minimum_nights float64 | NaN: 3 | with decimals: 0
maximum_nights float64 | NaN: 3 | with decimals: 0
minimum_minimum_nights float64 | NaN: 3 | with decimals: 0
maximum_minimum_nights float64 | NaN: 3 | with decimals: 0
minimum_maximum_nights float64 | NaN: 3 | with decimals: 0
maximum_maximum_nights float64 | NaN: 3 | with decimals: 0
rows with NaN in any: 3 | in all: 3


Convert to `Int64`.

In [50]:
for col in night_cols:
    listings_clean[col] = listings_clean[col].astype("Int64")

The dtype and the NaN after the conversion, and whether the sum of each column is still the same as in the raw table.

In [51]:
for col in night_cols:
    print(col, listings_clean[col].dtype, "| NaN:", listings_clean[col].isna().sum(),
          "| same sum as raw:", listings_clean[col].sum() == listings[col].sum())

minimum_nights Int64 | NaN: 3 | same sum as raw: True
maximum_nights Int64 | NaN: 3 | same sum as raw: True
minimum_minimum_nights Int64 | NaN: 3 | same sum as raw: True
maximum_minimum_nights Int64 | NaN: 3 | same sum as raw: True
minimum_maximum_nights Int64 | NaN: 3 | same sum as raw: True
maximum_maximum_nights Int64 | NaN: 3 | same sum as raw: True


**Findings:** the hypothesis holds.
- The 6 columns have 0 values with decimals, and the same 3 rows have NaN in all of them.
- All 6 are now `Int64` and still have 3 NaN, and the sum of each column is the same as in the raw table.
- The `*_avg_ntm` columns were left as float, because they are averages.

### 4g. Summary

**Hypothesis:** after 4a–4f every column has its planned dtype, no conversion created new NaN, the raw tables are unchanged, every id column is still `int64`, and `dataframes_clean` points to the latest `_clean` tables.

One row per converted column. For the three new columns, "before" describes the text column they come from (`bathrooms_text`, `amenities`).

In [52]:
conversions = [("listings", col, col) for col in date_cols + bool_cols + ["price"] + night_cols]
conversions += [
    ("listings", "bathrooms_text", "bathrooms"),
    ("listings", "bathrooms_text", "bath_shared"),
    ("listings", "amenities", "n_amenities"),
    ("calendar", "date", "date"),
    ("calendar", "available", "available"),
    ("reviews", "date", "date"),
]

rows = []
for table, col_before, col_after in conversions:
    before = dataframes[table][col_before]
    after = dataframes_clean[table][col_after]
    rows.append({
        "table": table,
        "column": col_after,
        "dtype_before": str(before.dtype),
        "dtype_after": str(after.dtype),
        "nan_before": before.isna().sum(),
        "nan_after": after.isna().sum()
    })

summary = pd.DataFrame(rows)
summary

,table,column,dtype_before,dtype_after,nan_before,nan_after
0,listings,last_scraped,str,datetime64[us],0,0
1,listings,calendar_last_scraped,str,datetime64[us],0,0
2,listings,first_review,str,datetime64[us],571,571
3,listings,last_review,str,datetime64[us],571,571
4,listings,price_quote_checkin_date,str,datetime64[us],300,300
5,listings,price_quote_checkout_date,str,datetime64[us],300,300
6,listings,host_is_superhost,str,boolean,0,0
7,listings,host_has_profile_pic,str,boolean,0,0
8,listings,host_identity_verified,str,boolean,0,0
9,listings,has_availability,str,boolean,11,11


The converted columns where the number of NaN changed.

In [53]:
summary[summary["nan_after"] != summary["nan_before"]]

,table,column,dtype_before,dtype_after,nan_before,nan_after


Compare each raw table with a fresh read of its CSV, to prove that it was not changed.

In [54]:
for name, df in dataframes.items():
    print(name, df.shape, "unchanged:", df.equals(pd.read_csv(f"data/{name}.csv")))

listings (4961, 90) unchanged: True
calendar (1810765, 5) unchanged: True
reviews (244369, 6) unchanged: True


The dtype of every id column in the `_clean` tables.

In [55]:
id_cols = ["id", "listing_id", "host_id", "host_profile_id", "reviewer_id"]
for name, df in dataframes_clean.items():
    print(name, df.shape, {col: str(df[col].dtype) for col in df.columns if col in id_cols})

listings (4961, 78) {'id': 'int64', 'host_id': 'int64', 'host_profile_id': 'int64'}
calendar (1810765, 5) {'listing_id': 'int64'}
reviews (244369, 6) {'listing_id': 'int64', 'id': 'int64', 'reviewer_id': 'int64'}


Check that the dict still points to the latest `_clean` tables.

In [56]:
print(dataframes_clean["listings"] is listings_clean,
      dataframes_clean["calendar"] is calendar_clean,
      dataframes_clean["reviews"] is reviews_clean)

True True True


**Findings:** the hypothesis holds.
- The table above has 23 rows: 8 date columns are `datetime64[us]`, 6 columns are `boolean` (5 flags and `bath_shared`), 6 are `Int64`, `price` and `bathrooms` are `float64`, and `n_amenities` is `int64`.
- The number of NaN changed in 0 of the 23 rows, so no conversion created new NaN.
- The raw tables are unchanged: all three are equal to a fresh read of their CSV, with shapes (4961, 90), (1810765, 5) and (244369, 6).
- All 7 id columns of the `_clean` tables are `int64`: `id`, `host_id` and `host_profile_id` in `listings`, `listing_id` in `calendar`, and `listing_id`, `id` and `reviewer_id` in `reviews`.
- `dataframes_clean` points to the latest `listings_clean`, `calendar_clean` and `reviews_clean`.
- No rows or columns were dropped. `listings_clean` went from 75 to 78 columns because of the 3 new ones: `bathrooms`, `bath_shared` and `n_amenities`.

## 5. Missing values by group

For each group of NaN: why is the value missing, and what do we do about it? This section mostly records. It changes no table.

The number of NaN in the three `_clean` tables now, to compare with the end of this section.

In [57]:
nan_start = {name: int(df.isna().sum().sum()) for name, df in dataframes_clean.items()}
nan_start

{'listings': 17263, 'calendar': 0, 'reviews': 14}

Every column of `listings_clean` that has NaN, using `column_summary` from Section 3.

In [58]:
nan_summary = column_summary(listings_clean)
nan_summary[nan_summary["n_nan"] > 0]

,dtype,n_nan,pct_nan,n_unique
source,str,4526,91.2,1
host_about,str,2750,55.4,622
host_location,str,1855,37.4,129
first_review,datetime64[us],571,11.5,1998
last_review,datetime64[us],571,11.5,709
review_scores_communication,float64,571,11.5,115
reviews_per_month,float64,571,11.5,567
review_scores_value,float64,571,11.5,138
review_scores_location,float64,571,11.5,121
review_scores_rating,float64,571,11.5,141


The same for `calendar_clean` and `reviews_clean`.

In [59]:
for name in ["calendar", "reviews"]:
    summary_one = column_summary(dataframes_clean[name])
    print(name, summary_one.loc[summary_one["n_nan"] > 0, "n_nan"].to_dict())

calendar {}
reviews {'comments': 14}


### 5a. Review columns (571 listings)

**Hypothesis:** these 571 listings have no reviews at all, so all 10 review columns are missing in exactly the same rows.

Count how many of the 10 review columns are missing in each row.

In [60]:
score_cols = [col for col in listings_clean.columns if col.startswith("review_scores_")]
review_cols = ["first_review", "last_review", "reviews_per_month"] + score_cols
print(len(review_cols), "columns")
listings_clean[review_cols].isna().sum(axis=1).value_counts()

10 columns


0     4390
10     571
Name: count, dtype: int64

Compare with `number_of_reviews`. Any row off the diagonal would be a listing with reviews but no scores, or the opposite.

In [61]:
no_review_values = listings_clean[review_cols].isna().all(axis=1).rename("no_review_values")
pd.crosstab(no_review_values, listings_clean["number_of_reviews"] == 0)

number_of_reviews,False,True
no_review_values,,
False,4390,0
True,0,571


The lowest value of each score column, and how many scores are 0.

In [62]:
print(listings_clean[score_cols].min().to_dict())
print("scores equal to 0:", (listings_clean[score_cols] == 0).sum().sum())

{'review_scores_rating': 1.0, 'review_scores_accuracy': 1.0, 'review_scores_cleanliness': 1.0, 'review_scores_checkin': 1.0, 'review_scores_communication': 1.0, 'review_scores_location': 1.0, 'review_scores_value': 1.0}
scores equal to 0: 0


**Findings:** the hypothesis holds.
- All 10 columns are missing together: 571 rows miss all 10 and 4,390 rows miss none.
- The 571 are exactly the listings with `number_of_reviews == 0`. Nothing is off the diagonal.
- The lowest value in all 7 score columns is 1.0, and no score is 0.

**Decision:** these columns stay NaN. A `review_scores_rating` of 0 would read as the worst possible score and pull every average down, while these listings have no score at all.

### 5b. Price columns (334 listings)

**Hypothesis:** a listing has no price when it could not be booked on the day of the scrape, so these listings should have 0 available days.

Count how many of the 7 price columns are missing in each row. Different counts mean different subgroups.

In [63]:
price_cols = ["price", "price_quote_checkin_date", "price_quote_checkout_date", "price_quote_total_price",
              "price_quote_price_per_night", "price_quote_raw", "estimated_revenue_l365d"]
price_nan_count = listings_clean[price_cols].isna().sum(axis=1).rename("price_nan_count")
price_nan_count.value_counts()

price_nan_count
0    4627
7     300
4      34
Name: count, dtype: int64

Which columns are missing in each subgroup.

In [64]:
listings_clean[price_cols].isna().groupby(price_nan_count).sum().T

price_nan_count,0,4,7
price,0,34,300
price_quote_checkin_date,0,0,300
price_quote_checkout_date,0,0,300
price_quote_total_price,0,34,300
price_quote_price_per_night,0,34,300
price_quote_raw,0,0,300
estimated_revenue_l365d,0,34,300


How many listings in each subgroup have 0 available days in the next 365.

In [65]:
pd.crosstab(price_nan_count, listings_clean["availability_365"] == 0)

availability_365,False,True
price_nan_count,,
0,4624,3
4,34,0
7,24,276


The `is_available` field inside `price_quote_raw` for the small subgroup, the 34 listings that have a quote but no price.

In [66]:
small_group = price_nan_count == 4
small_raw = listings_clean.loc[small_group, "price_quote_raw"]
is_available = small_raw.str.extract(r'"is_available": (\w+)')[0].rename("is_available")
is_available.value_counts()

is_available
false    23
null      6
true      5
Name: count, dtype: int64

One full `price_quote_raw` for each value of `is_available`.

In [67]:
for value in ["false", "null", "true"]:
    print(value, small_raw[is_available == value].iloc[0], "\n")

false {"quote": {"taxes": null, "currency": null, "date_match": null, "service_fee": null, "total_price": null, "cleaning_fee": null, "is_available": false, "discount_amount": null, "price_per_night": null, "nightly_subtotal": null, "is_monthly_priced": false, "discounted_subtotal": null, "raw_price_line_items": [], "returned_checkin_date": null, "requested_checkin_date": "2027-07-01", "returned_checkout_date": null, "requested_checkout_date": "2027-07-04"}} 

null {"quote": {"taxes": null, "currency": null, "date_match": null, "service_fee": null, "total_price": null, "cleaning_fee": null, "is_available": null, "discount_amount": null, "price_per_night": null, "nightly_subtotal": null, "is_monthly_priced": false, "discounted_subtotal": null, "raw_price_line_items": null, "returned_checkin_date": null, "requested_checkin_date": "2026-10-02", "returned_checkout_date": null, "requested_checkout_date": "2026-10-03"}} 

true {"quote": {"taxes": null, "currency": "EUR", "date_match": null, 

Which of the 34 quotes were priced per month, using the `is_monthly_priced` field.

In [68]:
is_monthly = small_raw.str.contains('"is_monthly_priced": true', regex=False).rename("is_monthly")
pd.crosstab(is_available, is_monthly)

is_monthly,False,True
is_available,,
false,23,0
null,6,0
true,0,5


The number of nights requested in the quote, for each value of `is_available`.

In [69]:
small_quotes = listings_clean.loc[small_group, ["id", "price_quote_checkin_date", "price_quote_checkout_date"]]
requested_nights = (small_quotes["price_quote_checkout_date"] - small_quotes["price_quote_checkin_date"]).dt.days.rename("nights")
pd.crosstab(is_available, requested_nights)

nights,1,2,3,4,7,28,50,60,61
is_available,,,,,,,,,
false,15,2,5,1,0,0,0,0,0
null,1,3,0,1,1,0,0,0,0
true,0,0,0,0,0,2,1,1,1


Are the requested nights free in `calendar`? Join the 34 quotes to their calendar rows, on a temporary table, and print the id dtypes after the merge.

In [70]:
quote_calendar = calendar_clean.merge(small_quotes, left_on="listing_id", right_on="id")
print(quote_calendar[["listing_id", "id"]].dtypes.to_dict())
in_stay = quote_calendar[(quote_calendar["date"] >= quote_calendar["price_quote_checkin_date"])
                         & (quote_calendar["date"] < quote_calendar["price_quote_checkout_date"])]
all_free = in_stay.groupby("listing_id")["available"].all()
all_free.value_counts()

{'listing_id': dtype('int64'), 'id': dtype('int64')}


available
True    34
Name: count, dtype: Int64

The free days of each listing in `calendar`, next to `availability_365`. The data dictionary says `availability_365` is determined by the calendar.

In [71]:
free_days = calendar_clean.groupby("listing_id")["available"].sum()
calendar_free = listings_clean["id"].map(free_days)
same_days = (calendar_free == listings_clean["availability_365"]).rename("same_as_availability_365")
pd.crosstab(price_nan_count, same_days)

same_as_availability_365,False,True
price_nan_count,,
0,3,4624
4,0,34
7,0,300


The subgroups compared with `source`, which is looked at in 5h.

In [72]:
pd.crosstab(price_nan_count, listings_clean["source"].isna())

source,False,True
price_nan_count,,
0,131,4496
4,5,29
7,299,1


The reviews held by each subgroup: what would be lost if these listings were dropped.

In [73]:
listings_clean.groupby(price_nan_count)["number_of_reviews"].sum()

price_nan_count
0    231403
4      2211
7     10755
Name: number_of_reviews, dtype: int64

**Findings:** the hypothesis holds only in part. There are two subgroups, and no price does not always mean no available days.

| subgroup | listings | missing columns | 0 available days | `source` is "previous scrape" | reviews |
|---|---|---|---|---|---|
| no quote at all | 300 | all 7 | 276 | 299 | 10,755 |
| quote without a price | 34 | 4 of 7 | 0 | 5 | 2,211 |

- **No quote (300):** `price_quote_raw` itself is missing. 276 have 0 available days and 24 have some, and 299 of the 300 are "previous scrape" listings (see 5h). The free days in `calendar` agree with `availability_365` for all 300.
- **Quote without a price (34):** `is_available` explains only part of them.
  - `false` for 23: Airbnb answered that the dates are not available, although `calendar` shows the requested nights free for all 34. The data does not show why.
  - `null` for 6: the quote holds no answer at all.
  - `true` for 5: all 5 are monthly-priced stays of 28 to 61 nights. The example above has a total inside `price_quote_raw`, but no price per night.
- Across all the listings, `calendar` and `availability_365` differ on only 3 rows. They are looked at in 5d.

**Decision:** the 334 listings stay, because dropping them would lose 12,966 reviews (10,755 + 2,211), and the NaN are not filled.

**For the EDA:** price statistics describe only the 4,627 listings with a price. A missing `estimated_revenue_l365d` is not a revenue of 0. The 0 available days do not prove that a listing is closed, because the data dictionary says a day may be booked or blocked.

### 5c. `has_availability` (11 NaN)

**Hypothesis:** NaN means that the listing has no availability, so these 11 listings should have 0 available days.

The availability columns of the 11 listings.

In [74]:
no_flag = listings_clean["has_availability"].isna()
listings_clean.loc[no_flag, ["id", "availability_30", "availability_60", "availability_90", "availability_365", "price"]]

,id,availability_30,availability_60,availability_90,availability_365,price
460,1481819772772785580,30,60,90,365,NaN
475,1550172868484251432,28,58,88,363,NaN
489,1596307758303162900,20,46,53,328,40.00
1250,35253879,19,49,79,354,85.50
1571,49433556,0,0,0,266,NaN
1998,814556924448179666,30,60,90,365,NaN
3401,1362336720354955658,24,54,84,359,63.75
3544,1409232005479150386,11,25,47,244,185.00
3679,1448466618425118854,6,30,60,328,74.00
4055,1557848414243399601,24,50,72,165,78.00


The same 11 in `calendar`: their free days next to `availability_365`.

In [75]:
(calendar_free[no_flag] == listings_clean.loc[no_flag, "availability_365"]).value_counts()

True    11
Name: count, dtype: Int64

The flag against 0 available days, over all listings.

In [76]:
pd.crosstab(listings_clean["has_availability"].astype(object).fillna("NaN"), listings_clean["availability_365"] == 0)

availability_365,False,True
has_availability,,
True,4671,279
NaN,11,0


**Findings:** the hypothesis is rejected.
- The 11 listings have 165 to 365 available days, so the NaN does not mean "no availability". Their free days in `calendar` agree with `availability_365` for all 11.
- 5 of the 11 have no price.
- The flag does not separate anything either: it is True for 279 listings with 0 available days, and it is never False.
- The data dictionary only says `[t=true; f=false]`, so it does not explain the NaN.

**Decision:** the 11 stay NaN, and they are not filled with True. The EDA should not rely on `has_availability`: `availability_365` says more, and it agrees with `calendar`.

### 5d. The `*_nights` columns (3 listings)

**Hypothesis:** the 3 listings found in 4f have something in common that explains why all 6 columns are missing.

Who they are: availability, reviews and price.

In [77]:
no_nights = listings_clean[night_cols].isna().all(axis=1)
listings_clean.loc[no_nights, ["id", "availability_30", "availability_90", "availability_365", "number_of_reviews", "last_review", "price"]]

,id,availability_30,availability_90,availability_365,number_of_reviews,last_review,price
471,1530520958886292391,0,0,0,32,2026-09-21,87.0
3008,1264320835682122596,0,0,0,385,2026-09-22,74.6
4486,1655270337043787142,0,0,0,5,2026-09-23,60.0


Are they the only listings with a price and 0 available days?

In [78]:
priced_no_days = listings_clean["price"].notna() & (listings_clean["availability_365"] == 0)
print("priced with 0 days:", priced_no_days.sum(), "| same rows as the 3:", priced_no_days.equals(no_nights))

priced with 0 days: 3 | same rows as the 3: True


Their free days and their minimum nights in `calendar`, next to `availability_365`.

In [79]:
three_ids = listings_clean.loc[no_nights, "id"]
calendar_clean[calendar_clean["listing_id"].isin(three_ids)].groupby("listing_id").agg(
    calendar_rows=("date", "size"), free_days=("available", "sum"), median_min_nights=("minimum_nights", "median"))

,calendar_rows,free_days,median_min_nights
listing_id,,,
1264320835682122596,365,365,1.0
1530520958886292391,365,335,1.0
1655270337043787142,365,347,1.0


**Findings:** the 3 listings have a lot in common.
- They have 0 in `availability_30`, `availability_90` and `availability_365`, and all 6 `*_nights` columns are missing.
- They are not inactive: they have a price, 32, 385 and 5 reviews, and their last reviews are from 21 to 23 September 2026, just before the scrape.
- They are the only listings with a price and 0 available days.
- `calendar` disagrees with their `availability_365` of 0: it has 365 rows for each, with 365, 335 and 347 free days and a median minimum stay of 1 night.

So the problem seems to be in these fields of `listings`, not in the listings themselves, but the data does not show the cause.

**Decision:** the 3 stay NaN and are not filled from `calendar`. Their `availability_*` values should not be trusted. The disagreement goes to Section 7 (cross-table consistency).

### 5e. `license` (105 NaN), and the values written differently

**Hypothesis:** in Greece a listing needs a registration number, so the NaN are not by design and should concentrate in a particular kind of listing.

The lengths of the values, with NaN counted.

In [80]:
license_len = listings_clean["license"].str.len()
license_len.value_counts(dropna=False)

license
11.0    4404
7.0      303
16.0     112
NaN      105
6.0       37
Name: count, dtype: int64

One example of each length.

In [81]:
for length in sorted(license_len.dropna().unique()):
    print(int(length), listings_clean.loc[license_len == length, "license"].iloc[0])

6 Exempt
7 3672670
11 00003427019
16 0933K274A0815400


The only value that is a word.

In [82]:
exempt = listings_clean["license"] == "Exempt"
print("listings:", exempt.sum(), "| hosts:", listings_clean.loc[exempt, "host_id"].nunique())
listings_clean.loc[exempt, "room_type"].value_counts()

listings: 37 | hosts: 22


room_type
Entire home/apt    33
Private room        4
Name: count, dtype: int64

The 7-digit and the 11-digit codes: is the 11-digit form the same number with leading zeros? Count the codes that appear in both forms.

In [83]:
codes_7 = listings_clean.loc[license_len == 7, "license"]
codes_11 = listings_clean.loc[license_len == 11, "license"]
print("11 digits starting with 0000:", codes_11.str.startswith("0000").sum(), "of", len(codes_11))
print("in both forms:", set(codes_7) & set(codes_11.str.lstrip("0")))

11 digits starting with 0000: 4368 of 4404
in both forms: {'2317911'}


The 16-character codes look alike, but the letters may not be the same. `ord` gives the code point of a character: 75 is the Latin K and 922 the Greek Κ, 65 the Latin A and 913 the Greek Α.

In [84]:
codes_16 = listings_clean.loc[license_len == 16, "license"]
print("position 5:", codes_16.str[4].map(ord).value_counts().to_dict())
print("position 9:", codes_16.str[8].map(ord).value_counts().to_dict())

position 5: {922: 89, 75: 23}
position 9: {48: 68, 75: 15, 922: 15, 913: 10, 65: 4}


Do the two alphabets split the same code in two? Replace the Greek letters with Latin ones on a temporary Series, then count the distinct codes.

In [85]:
latin_only = codes_16.str.replace("\u039a", "K").str.replace("\u0391", "A")
print("distinct as written:", codes_16.nunique(), "| distinct in one alphabet:", latin_only.nunique())
ways_written = codes_16.groupby(latin_only).nunique()
print(ways_written[ways_written > 1])

distinct as written: 28 | distinct in one alphabet: 25
license
0933K131K0099400    2
0933K134K0099300    2
0933K134K0772401    2
Name: license, dtype: int64


The first four characters of the 16-character codes.

In [86]:
codes_16.str[:4].value_counts()

license
0933    111
0938      1
Name: count, dtype: int64

The share of NaN in each room type.

In [87]:
no_license = listings_clean["license"].isna()
pd.DataFrame({
    "n_nan": no_license.groupby(listings_clean["room_type"]).sum(),
    "pct_nan": (no_license.groupby(listings_clean["room_type"]).mean() * 100).round(1)
})

,n_nan,pct_nan
room_type,,
Entire home/apt,102,2.1
Private room,3,2.1
Shared room,0,0.0


Do the NaN come from particular hosts? Count the hosts, the largest counts, and the share of listings whose host has only 1 listing.

In [88]:
print("hosts with a NaN:", listings_clean.loc[no_license, "host_id"].nunique())
print("largest counts:", listings_clean.loc[no_license, "host_id"].value_counts().head(3).tolist())
print("largest host overall:", listings_clean["calculated_host_listings_count"].max(), "listings")
one_listing = listings_clean["calculated_host_listings_count"] == 1
print("host with 1 listing, % of NaN:", round(one_listing[no_license].mean() * 100, 1),
      "| % of all listings:", round(one_listing.mean() * 100, 1))

hosts with a NaN: 86
largest counts: [5, 4, 4]
largest host overall: 212 listings
host with 1 listing, % of NaN: 26.7 | % of all listings: 31.2


**Findings:** the second half of the hypothesis is rejected: the NaN do not concentrate in one kind of listing.
- **NaN (105):** 2.1% of `Entire home/apt` and 2.1% of `Private room`, and 0 `Shared room`. They come from 86 hosts, at most 5 from one host. 26.7% of them belong to a host with 1 listing, against 31.2% of all listings. The data does not show why they are missing, so "not by design" remains an assumption.
- **Lengths:** 11 characters for 4,404 values, 7 for 303, 16 for 112 and 6 for 37.
- **The word:** the 6-character value is `Exempt`, in 37 listings from 22 hosts. It most likely means that the host states the listing is exempt from registration, which the data cannot confirm. It is a stated answer, so it is not the same as a NaN.
- **Digits:** 4,368 of the 4,404 11-digit codes start with `0000`, but only 1 code appears in both the 7-digit and the 11-digit form (`2317911`). So most 7-digit codes cannot be matched to a longer one.
- **The hidden error in the 16-character codes:** the letters look identical but come from two alphabets. Position 5 is the Greek Κ in 89 values and the Latin K in 23. Position 9 is also affected: Latin K 15, Greek Κ 15, Greek Α 10, Latin A 4, and the digit 0 in the other 68. Because of this, 3 codes are written in two ways: 28 distinct codes become 25 in one alphabet.
- **Prefix:** 111 of the 16-character codes start with `0933` and 1 with `0938`. Whether that one is a typo is not known yet.

**Decision:** the 105 stay NaN and `Exempt` stays as written. The two alphabets and the `0938` code are recorded here and go to Section 8 (typos).

### 5f. `beds` (48 NaN) and `bathrooms_text` (1 NaN)

**Hypothesis:** `bedrooms` and `accommodates` are filled for these 48 listings, and they may give a value of `beds` that can be defended.

`bedrooms` against `accommodates` for the 48 listings.

In [89]:
no_beds = listings_clean["beds"].isna()
print("room types:", listings_clean.loc[no_beds, "room_type"].value_counts().to_dict())
pd.crosstab(listings_clean.loc[no_beds, "bedrooms"], listings_clean.loc[no_beds, "accommodates"])

room types: {'Entire home/apt': 48}


accommodates,2,3,4,5,6,7
bedrooms,,,,,,
0,4,1,1,0,0,0
1,14,4,9,1,0,0
2,0,1,4,0,5,2
3,0,0,0,1,0,1


How much `beds` varies among the other listings, for the two most common values of `accommodates` among the 48.

In [90]:
for guests in [2, 4]:
    share = listings_clean.loc[listings_clean["accommodates"] == guests, "beds"].value_counts(normalize=True)
    print(guests, "guests:", (share.head(3) * 100).round(1).to_dict())

2 guests: {1.0: 87.1, 2.0: 11.7, 3.0: 0.7}
4 guests: {2.0: 66.4, 1.0: 20.5, 3.0: 10.7}


The same with `bedrooms` fixed too, at 1 bedroom, the most common value among the 48.

In [91]:
for guests in [2, 4]:
    same_size = (listings_clean["accommodates"] == guests) & (listings_clean["bedrooms"] == 1)
    share = listings_clean.loc[same_size, "beds"].value_counts(normalize=True)
    print(guests, "guests, 1 bedroom:", same_size.sum(), "listings,", (share.head(3) * 100).round(1).to_dict())

2 guests, 1 bedroom: 1161 listings, {1.0: 87.7, 2.0: 11.6, 3.0: 0.5}
4 guests, 1 bedroom: 1087 listings, {2.0: 67.3, 1.0: 27.1, 3.0: 4.4}


The 1 listing with no `bathrooms_text`, which is also missing in `bathrooms` and `bath_shared` since 4d.

In [92]:
listings_clean.loc[listings_clean["bathrooms_text"].isna(), ["id", "bathrooms_text", "bathrooms", "bath_shared",
                                                               "room_type", "bedrooms", "accommodates", "number_of_reviews"]]

,id,bathrooms_text,bathrooms,bath_shared,room_type,bedrooms,accommodates,number_of_reviews
3732,1463078291064408142,NaN,NaN,<NA>,Entire home/apt,1,2,10


**Findings:** the hypothesis does not hold: `bedrooms` and `accommodates` are filled, but they do not give a value that can be defended.
- All 48 listings are `Entire home/apt`. They have a `bedrooms` value (0 for 6 of them) and an `accommodates` value.
- For 2 guests and 1 bedroom, 87.7% of the other listings have 1 bed and 11.6% have 2.
- For 4 guests and 1 bedroom, 67.3% have 2 beds and 27.1% have 1, so the most common value would be wrong for about 1 in 3 listings.
- The 1 listing with no `bathrooms_text` (`id` 1463078291064408142) has 1 bedroom, 2 guests and 10 reviews. None of these columns says what its bathroom is.

**Decision:** `beds` stays NaN for the 48. `bathrooms_text`, `bathrooms` and `bath_shared` stay NaN for the 1 listing.

### 5g. Free text

**Hypothesis:** by design: the person simply did not write anything, so nothing about these NaN needs fixing.

The NaN of each text column, and how many filled values are only blanks.

In [93]:
for col in ["host_about", "host_location", "description"]:
    text = listings_clean[col]
    print(col, "| NaN:", text.isna().sum(), "| blank:", (text.dropna().str.strip() == "").sum())
comments = reviews_clean["comments"]
print("comments | NaN:", comments.isna().sum(), "| blank:", (comments.dropna().str.strip() == "").sum())

host_about | NaN: 2750 | blank: 0
host_location | NaN: 1855 | blank: 0
description | NaN: 1 | blank: 0
comments | NaN: 14 | blank: 0


`host_about` and `host_location` describe the host, so check whether the NaN is the same across all the listings of one host. A maximum of 1 means that no host is mixed.

In [94]:
hosts = listings_clean["host_id"]
print("hosts:", hosts.nunique())
for col in ["host_about", "host_location"]:
    print(col, "| hosts with NaN:", hosts[listings_clean[col].isna()].nunique(),
          "| most states per host:", listings_clean[col].isna().groupby(hosts).nunique().max())

hosts: 2122
host_about | hosts with NaN: 1492 | most states per host: 1
host_location | hosts with NaN: 856 | most states per host: 1


A few filled values, shortened to 60 characters.

In [95]:
for col in ["host_about", "host_location", "description"]:
    print(col, listings_clean[col].dropna().sample(3, random_state=1).str[:60].tolist())

host_about ['Με όραμα την Ελληνική Φιλοξενία και μοναδικό σκοπό να προσφέ', 'At halu! we combine the best qualities of Hotels and BnBs. W', 'Established in Thessaloniki, Greece, HOUSELOFT specializes i']
host_location ['Thessaloniki, Greece', 'Thessaloniki, Greece', 'Thessaloniki, Greece']
description ['A warm farmhouse-style apartment in Ano Poli with a private ', 'A modern apartment in the city center that stands out with i', 'Enjoy a stylish experience at this centrally-located place.']


The 1 listing with no `description`.

In [96]:
listings_clean.loc[listings_clean["description"].isna(), ["id", "name", "room_type", "number_of_reviews"]]

,id,name,room_type,number_of_reviews
304,31925787,Apartment in nice building (Fully equipped),Entire home/apt,0


All 14 reviews with no comment. The reviewer's name is left out, because it does not help here.

In [97]:
reviews_clean.loc[comments.isna(), ["listing_id", "id", "date", "comments"]]

,listing_id,id,date,comments
5741,2417714,897684484074412311,2023-05-23,NaN
11140,4530300,1014350411194330342,2023-10-31,NaN
11711,12820336,362356254,2018-12-26,NaN
21807,8214869,1678193368549732708,2026-05-04,NaN
25728,14246742,1530394593722719713,2025-10-12,NaN
36128,16530054,436427291,2019-04-13,NaN
57294,23774911,1611554229005449914,2026-02-01,NaN
63461,24352142,1146174531957168030,2024-04-30,NaN
100175,34191893,1011378203495062150,2023-10-27,NaN
121981,45613610,1480380033614022526,2025-08-04,NaN


Comments that are not NaN but say almost nothing: 2 characters or fewer.

In [98]:
short_comments = comments[comments.str.len() <= 2]
print("comments with 2 characters or fewer:", len(short_comments))
short_comments.value_counts().head(5)

comments with 2 characters or fewer: 3202


comments
.     1650
👍      154
-      135
!       92
Ok      63
Name: count, dtype: int64

**Findings:** the hypothesis holds.
- `host_about` has 2,750 NaN, from 1,492 of the 2,122 hosts, and `host_location` has 1,855, from 856 hosts. No host is mixed: all the listings of a host are either filled or missing, which fits a field that the host left empty.
- `description` has 1 NaN (`id` 31925787, 0 reviews).
- `reviews.comments` has 14 NaN, in 14 reviews from 2018 to 2026. There is no text to read in them, so the table above shows only which reviews they are.
- No text column has a filled value made only of blanks.
- 3,202 comments have 2 characters or fewer and say almost nothing: `.` alone appears 1,650 times. They are not NaN, so they are left for Section 8.

**Decision:** all four columns stay NaN. No placeholder text is added.

### 5h. `source` (435 "previous scrape")

The data dictionary (version 4.3) says: *"One of "neighbourhood search" or "previous scrape". "neighbourhood search" means that the listing was found by searching the city, while "previous scrape" means that the listing was seen in another scrape performed in the last 65 days, and the listing was confirmed to be still available on the Airbnb site."*

**Hypothesis:** a "previous scrape" listing was not found by the search on this run, so it more often looks inactive: no reviews, no price, and no available days.

The values of `source`, with NaN counted. In this file the other value of the dictionary, "neighbourhood search", does not appear.

In [99]:
listings_clean["source"].value_counts(dropna=False)

source
NaN                4526
previous scrape     435
Name: count, dtype: int64

`source` against no reviews, no price and 0 available days: the counts, then the share of each `source` group. `pd.crosstab` drops NaN by default, so `source` is filled on a temporary Series.

In [100]:
source_group = listings_clean["source"].fillna("NaN")
checks = {
    "no reviews": listings_clean["number_of_reviews"] == 0,
    "no price": listings_clean["price"].isna(),
    "0 available days": listings_clean["availability_365"] == 0,
}
for label, flag in checks.items():
    print(label)
    print(pd.crosstab(source_group, flag))
    print(pd.crosstab(source_group, flag, normalize="index").round(3), "\n")

no reviews
number_of_reviews  False  True 
source                         
NaN                 4033    493
previous scrape      357     78
number_of_reviews  False  True 
source                         
NaN                0.891  0.109
previous scrape    0.821  0.179 

no price
price            False  True 
source                       
NaN               4496     30
previous scrape    131    304
price            False  True 
source                       
NaN              0.993  0.007
previous scrape  0.301  0.699 

0 available days
availability_365  False  True 
source                        
NaN                4522      4
previous scrape     160    275
availability_365  False  True 
source                        
NaN               0.999  0.001
previous scrape   0.368  0.632 



**Findings:** the hypothesis holds, and the link to reviews is the weakest of the three.

| check | `previous scrape` (435) | NaN (4,526) |
|---|---|---|
| no reviews | 78 (17.9%) | 493 (10.9%) |
| no price | 304 (69.9%) | 30 (0.7%) |
| 0 available days | 275 (63.2%) | 4 (0.1%) |

- The dictionary names a second value, "neighbourhood search", but this file never uses it. The NaN probably stand for it, but neither the file nor the dictionary says so.
- "Previous scrape" does not mean closed: the dictionary says these listings were confirmed to be still on the site, and 131 of them have a price.

**Decision:** `source` stays, because it marks most of the listings with no price or no available days. The NaN are not filled.

### 5i. Summary

**Hypothesis:** this section changed no table: the NaN are the same as at its start, and the id columns are still `int64`.

The NaN of the three tables against the start of the section, and the raw tables against a fresh read of their CSV.

In [101]:
for name, df in dataframes_clean.items():
    print(name, df.shape, "| NaN now:", df.isna().sum().sum(), "| at the start:", nan_start[name],
          "| raw unchanged:", dataframes[name].equals(pd.read_csv(f"data/{name}.csv")))

listings (4961, 78) | NaN now: 17263 | at the start: 17263 | raw unchanged: True
calendar (1810765, 5) | NaN now: 0 | at the start: 0 | raw unchanged: True
reviews (244369, 6) | NaN now: 14 | at the start: 14 | raw unchanged: True


The id columns of the `_clean` tables, and `dataframes_clean` against the latest tables.

In [102]:
id_cols = ["id", "listing_id", "host_id", "host_profile_id", "reviewer_id"]
for name, df in dataframes_clean.items():
    print(name, df[df.columns.intersection(id_cols)].dtypes.astype(str).to_dict())
print(dataframes_clean["listings"] is listings_clean,
      dataframes_clean["calendar"] is calendar_clean,
      dataframes_clean["reviews"] is reviews_clean)

listings {'id': 'int64', 'host_id': 'int64', 'host_profile_id': 'int64'}
calendar {'listing_id': 'int64'}
reviews {'listing_id': 'int64', 'id': 'int64', 'reviewer_id': 'int64'}
True True True


**Findings:** the hypothesis holds. The NaN are the same as at the start of the section (17,263, 0 and 14), the raw tables equal a fresh read of their CSV, all 7 id columns are `int64`, and `dataframes_clean` points to the latest tables.

| group | columns | NaN | why they are missing | decision |
|---|---|---|---|---|
| reviews | `first_review`, `last_review`, `reviews_per_month`, 7 `review_scores_*` | 571 | the listing has no reviews | stay NaN |
| price, no quote | `price`, all 5 `price_quote_*`, `estimated_revenue_l365d` | 300 | no quote exists; mostly "previous scrape" listings with 0 available days | stay NaN, rows kept |
| price, with a quote | `price`, `price_quote_total_price`, `price_quote_price_per_night`, `estimated_revenue_l365d` | 34 | 23 `is_available` false, 6 null, 5 monthly-priced | stay NaN, rows kept |
| `has_availability` | 1 | 11 | not explained; these listings have available days | stay NaN, do not rely on the flag |
| `*_nights` | 6 | 3 | not explained; `availability_365` also disagrees with `calendar` | stay NaN, check in Section 7 |
| `license` | 1 | 105 | not explained; no particular kind of listing | stay NaN; typos in Section 8 |
| `beds` | 1 | 48 | not explained; no defensible value | stay NaN |
| bathrooms | `bathrooms_text`, `bathrooms`, `bath_shared` | 1 | not explained | stay NaN |
| free text | `host_about`, `host_location`, `description`, `reviews.comments` | 2,750, 1,855, 1, 14 | the person did not write anything | stay NaN |
| `source` | 1 | 4,526 | the file seems to leave the other value empty | stay NaN, column kept |

Section 5 filled 0 values. The EDA should know that price statistics cover only the listings with a price, that a missing value in these columns is not a 0, and that `has_availability` should not be relied on.

## 6. Keys, duplicates and referential integrity

Are the keys of Section 2 really unique, is anything entered twice, and does every row point to a listing that exists? This section records. It changes no table.

### 6a. The declared keys

**Hypothesis:** `listings.id`, (`calendar.listing_id`, `calendar.date`) and `reviews.id` are unique, and no key has NaN.

A dict of table and key, and a loop. `duplicated(subset=..., keep=False)` gives True to every row that shares its key with another row.

In [103]:
table_keys = {"listings": ["id"], "calendar": ["listing_id", "date"], "reviews": ["id"]}
for name, key in table_keys.items():
    df = dataframes_clean[name]
    print(name, key, "| rows sharing the key:", df.duplicated(subset=key, keep=False).sum(),
          "| NaN in the key:", df[key].isna().sum().sum())

listings ['id'] | rows sharing the key: 0 | NaN in the key: 0
calendar ['listing_id', 'date'] | rows sharing the key: 0 | NaN in the key: 0
reviews ['id'] | rows sharing the key: 0 | NaN in the key: 0


Whole rows that appear more than once, using `duplicated()` with no subset.

In [104]:
for name, df in dataframes_clean.items():
    print(name, "fully duplicated rows:", df.duplicated(keep=False).sum())

listings fully duplicated rows: 0
calendar fully duplicated rows: 0
reviews fully duplicated rows: 0


**Findings:** the hypothesis holds.
- No row shares its key with another row in any of the three tables, and no key has NaN.
- No table has a fully duplicated row.

**Decision:** the status in the grain table of Section 2 is changed from `assumed` to `verified`. No rows are dropped.

### 6b. `calendar`: 365 rows per listing, but 366 dates

**Hypothesis:** each listing's calendar starts on the day that listing was scraped, and the listings were scraped on two different days, which gives 366 dates in total.

How many rows each listing has.

In [105]:
calendar_clean.groupby("listing_id").size().value_counts()

365    4961
Name: count, dtype: int64

The first date of each listing, and how many listings start on each date.

In [106]:
first_date = calendar_clean.groupby("listing_id")["date"].min()
first_date.value_counts().sort_index()

date
2026-09-23      21
2026-09-24    4940
Name: count, dtype: int64

The first date of each listing against its `last_scraped`.

In [107]:
pd.crosstab(listings_clean["id"].map(first_date).rename("first_date"), listings_clean["last_scraped"])

last_scraped,2026-09-23,2026-09-24
first_date,,
2026-09-23,21,0
2026-09-24,0,4940


The last date of each listing, and how many days lie between its first and last date.

In [108]:
last_date = calendar_clean.groupby("listing_id")["date"].max()
print(last_date.value_counts().sort_index().to_dict())
(last_date - first_date).dt.days.value_counts()

{Timestamp('2027-09-22 00:00:00'): 21, Timestamp('2027-09-23 00:00:00'): 4940}


date
364    4961
Name: count, dtype: int64

Are any days missing inside a listing's calendar? `diff` gives the time since the previous row, so every value should be 1 day.

In [109]:
calendar_clean.sort_values(["listing_id", "date"]).groupby("listing_id")["date"].diff().dt.days.value_counts()

date
1.0    1805804
Name: count, dtype: int64

The number of listings on the first two and the last two dates.

In [110]:
calendar_clean["date"].value_counts().sort_index().iloc[[0, 1, -2, -1]]

date
2026-09-23      21
2026-09-24    4961
2027-09-22    4961
2027-09-23    4940
Name: count, dtype: int64

**Findings:** the hypothesis holds.
- All 4,961 listings have exactly 365 rows.
- 21 listings start on 2026-09-23 and 4,940 on 2026-09-24, and for every listing the first date equals its `last_scraped`.
- This explains the 366: 2026-09-23 appears only for the 21 listings, and 2027-09-23 only for the 4,940. The 364 dates between them appear for all listings (4,961 rows on 2026-09-24 and on 2027-09-22).
- It is not a year in the sense of "from a date to the same date next year": the first and the last date are 364 days apart, so the calendar ends the day before the anniversary. It does cover 365 days in a row, with no day missing.

**Decision:** nothing to fix. When comparing dates across listings, remember that 21 listings are one day earlier.

### 6c. `reviews`: the same person, the same home, the same day

**Hypothesis:** `reviews.id` is unique (6a), but a review entered twice would have a new id. So rows with the same (`listing_id`, `reviewer_id`, `date`) would be double entries, and they should be very few.

Count the rows with the same listing, reviewer and date.

In [111]:
same_day_key = ["listing_id", "reviewer_id", "date"]
same_day = reviews_clean[reviews_clean.duplicated(subset=same_day_key, keep=False)].sort_values(same_day_key)
print("rows:", len(same_day), "| groups:", same_day.groupby(same_day_key).ngroups)
print("rows with the same comment too:", reviews_clean.duplicated(subset=same_day_key + ["comments"], keep=False).sum())

rows: 4 | groups: 2
rows with the same comment too: 0


All of them, with the full comment. `itertuples` goes through a table one row at a time.

In [112]:
for row in same_day.itertuples():
    print(row.listing_id, row.id, row.date.date(), "|", row.comments)

14246128 672048398 2020-10-07 | Καθαρος χωρος  μεσα στο κεντρο της πολης και ωραια διακοσμημενος χωρος
14246128 672054275 2020-10-07 | Καθαρος χωρος με ζεστη ατμοσφαιρα και στο κεντρο της πολης
1375215036620861515 1417978943954566813 2025-05-10 | A quiet and good place to stay.
1375215036620861515 1418067371816604843 2025-05-10 | A wonderful experience in the heart of Thessaloniki! The hosts were extremely welcoming, the room was comfortable, and the location was perfect for exploring the city on foot. We felt right at home and will definitely be back!


One row per listing and reviewer, with their number of reviews, distinct days, and first and last date. Then keep the pairs with more than one review.

In [113]:
pairs = reviews_clean.groupby(["listing_id", "reviewer_id"]).agg(
    n_reviews=("id", "size"), n_days=("date", "nunique"), first=("date", "min"), last=("date", "max"))
repeat_pairs = pairs[pairs["n_reviews"] > 1]
print("pairs:", len(pairs), "| with more than one review:", len(repeat_pairs), "| on different days:", (repeat_pairs["n_days"] > 1).sum())
print("reviewers:", repeat_pairs.index.get_level_values("reviewer_id").nunique(), "| review rows:", repeat_pairs["n_reviews"].sum())

pairs: 238840 | with more than one review: 4391 | on different days: 4389
reviewers: 3848 | review rows: 9920


How many reviews these pairs have, and how many days lie between their first and last review.

In [114]:
print(repeat_pairs["n_reviews"].value_counts().sort_index().to_dict())
(repeat_pairs["last"] - repeat_pairs["first"]).dt.days.describe().round(1)

{2: 3714, 3: 455, 4: 124, 5: 42, 6: 27, 7: 11, 8: 5, 9: 3, 10: 5, 12: 2, 13: 1, 14: 2}


count    4391.0
mean      330.7
std       435.3
min         0.0
25%        31.0
50%       178.0
75%       400.5
max      3521.0
dtype: float64

A guest who returns would normally write a new text. Count the reviews whose text is the same as another review by the same person for the same listing.

In [115]:
with_text = reviews_clean[reviews_clean["comments"].notna()]
copied = with_text[with_text.duplicated(subset=["listing_id", "reviewer_id", "comments"], keep=False)]
print("rows:", len(copied), "| 50 characters or more:", (copied["comments"].str.len() >= 50).sum())
copied["comments"].value_counts().head(3)

rows: 327 | 50 characters or more: 46


comments
Τέλεια                 86
.                      66
Everything was good     9
Name: count, dtype: int64

The long copies: how many pairs, the days between the copies, and two examples shortened to 70 characters.

In [116]:
long_copied = copied[copied["comments"].str.len() >= 50].sort_values(["listing_id", "reviewer_id", "date"])
copy_days = long_copied.groupby(["listing_id", "reviewer_id", "comments"])["date"].agg(["min", "max"])
print("groups:", len(copy_days))
print(((copy_days["max"] - copy_days["min"]).dt.days).describe().round(1).to_dict())
for row in long_copied.head(4).itertuples():
    print(row.listing_id, row.date.date(), "|", row.comments[:70])

groups: 23
{'count': 23.0, 'mean': 5.6, 'std': 6.3, 'min': 1.0, '25%': 1.0, '50%': 3.0, '75%': 8.0, 'max': 28.0}
2788858 2023-03-12 | It was great to stay at this apartment.The flat was so clean, have a l
2788858 2023-04-09 | It was great to stay at this apartment.The flat was so clean, have a l
7609332 2024-10-19 | Hieno paikka keskustassa kävelymatkan päässä kaikesta. Olimme 6 aikuis
7609332 2024-10-22 | Hieno paikka keskustassa kävelymatkan päässä kaikesta. Olimme 6 aikuis


**Findings:** the hypothesis holds only in its size. Few rows match, but they do not look like plain double entries.
- **Same listing, reviewer and date:** 4 rows in 2 groups, and none has the same comment. In the first group, both texts say the same things in other words, so it may be one review written twice. In the second, a short text and a long one are quite different. The data cannot tell whether these are two stays.
- **More than one review on the same listing:** 4,391 of the 238,840 listing and reviewer pairs, from 3,848 reviewers, in 9,920 rows. 4,389 pairs are on different days. Most have 2 reviews (3,714), the most is 14, and the median time from first to last review is 178 days. Guests do return to the same home, so this looks normal.
- **What the plan did not expect:** 327 reviews have the same text as another review by the same person for the same listing. Most are short (`Τέλεια` 86, `.` 66), which can happen by chance. But 46 have 50 characters or more, in 23 groups of 2, and the copies are only 1 to 28 days apart (median 3). The dates differ, so the check above does not catch them. They could be double entries, or a guest pasting the old text after a new stay.

**Decision:** all rows stay. The 2 same-day groups are kept, because their texts differ. Repeat reviewers are kept, because they are normal. The 23 long copies are kept and recorded: at most 23 rows out of 244,369 are extra, and the data does not prove that they are errors.

### 6d. `listings`: the same home twice?

The Inside Airbnb *Data Assumptions* page says: *"Location information for listings are anonymized by Airbnb. In practice, this means the location for a listing on the map, or in the data will be from 0-450 feet (150 metres) of the actual address."* It also says: *"Listings in the same building are anonymized by Airbnb individually, and therefore may appear "scattered" in the area surrounding the actual address."*

**Hypothesis:** many listings share a host and their coordinates or their name, but they are different homes of that host, not the same home listed twice.

Listings with the same host and exactly the same coordinates, and with the same coordinates and any host.

In [117]:
same_place_key = ["host_id", "latitude", "longitude"]
same_place = listings_clean[listings_clean.duplicated(subset=same_place_key, keep=False)]
print("same host and coordinates:", len(same_place), "listings in", same_place.groupby(same_place_key).ngroups, "groups")
print("same coordinates, any host:", listings_clean.duplicated(subset=["latitude", "longitude"], keep=False).sum())

same host and coordinates: 804 listings in 226 groups
same coordinates, any host: 962


Listings with the same host and the same name.

In [118]:
same_name_key = ["host_id", "name"]
same_name = listings_clean[listings_clean.duplicated(subset=same_name_key, keep=False)]
print("same host and name:", len(same_name), "listings in", same_name.groupby(same_name_key).ngroups, "groups")

same host and name: 108 listings in 32 groups


Stricter and stricter matches: same host, coordinates and name, then also the same room type, guests and price, then also the same number of reviews.

In [119]:
both_key = same_place_key + ["name"]
print("host, coordinates, name:", listings_clean.duplicated(subset=both_key, keep=False).sum())
offer_key = both_key + ["room_type", "accommodates", "bedrooms", "price"]
print("and room type, guests, bedrooms, price:", listings_clean.duplicated(subset=offer_key, keep=False).sum())
print("and number of reviews:", listings_clean.duplicated(subset=offer_key + ["number_of_reviews"], keep=False).sum())

host, coordinates, name: 44
and room type, guests, bedrooms, price: 38
and number of reviews: 2


Examples side by side: same coordinates, same name, and both.

In [120]:
show_cols = ["id", "name", "latitude", "longitude", "room_type", "accommodates", "price", "number_of_reviews"]
print(same_place.sort_values(same_place_key)[show_cols].head(2).to_string(), "\n")
print(same_name.sort_values(same_name_key)[show_cols].head(3).to_string(), "\n")
same_both = listings_clean[listings_clean.duplicated(subset=both_key, keep=False)]
print(same_both.sort_values(both_key)[show_cols].head(3).to_string())

                      id        name   latitude  longitude        room_type  accommodates  price  number_of_reviews
2277  963545448125905545  Ursa Minor  40.596508  22.953918  Entire home/apt             4   70.0                 57
2278  963577323616360594  Ursa Major  40.596508  22.953918  Entire home/apt             4    NaN                 47 

                       id                               name   latitude  longitude        room_type  accommodates   price  number_of_reviews
1931   763698320578684092  F & B Collection - 2 Bedroom Flat  40.640161  22.937367  Entire home/apt             4   86.50                 24
2774  1162905538150174038  F & B Collection - 2 Bedroom Flat  40.571977  22.957392  Entire home/apt             6  110.50                 29
3830  1493328104796840743  F & B Collection - 2 Bedroom Flat  40.619002  22.955129  Entire home/apt             4   80.33                 12 

                       id                                       name   latitude  lon

The 2 listings that match on everything above are the strongest candidates for one home listed twice. Do they also share their reviews and their calendar?

In [121]:
closest = listings_clean[listings_clean.duplicated(subset=offer_key + ["number_of_reviews"], keep=False)]
print(closest[["id", "name", "price", "number_of_reviews", "first_review", "last_review"]].to_string())
closest_ids = closest["id"].tolist()
closest_reviews = reviews_clean[reviews_clean["listing_id"].isin(closest_ids)]
print("review ids shared:", closest_reviews["id"].duplicated().sum(), "| reviewers in both:",
      (closest_reviews.groupby("reviewer_id")["listing_id"].nunique() > 1).sum())
closest_calendar = calendar_clean[calendar_clean["listing_id"].isin(closest_ids)]
print("dates with the same availability:", (closest_calendar.groupby("date")["available"].nunique() == 1).sum(), "of", closest_calendar["date"].nunique())
print("free days:", closest_calendar.groupby("listing_id")["available"].sum().to_dict())

                       id               name  price  number_of_reviews first_review last_review
2399  1016487917711700538  DELUXE ΔΙΑΜΕΡΙΣΜΑ  108.0                  3   2024-02-03  2026-02-15
2429  1035296111642091276  DELUXE ΔΙΑΜΕΡΙΣΜΑ  108.0                  3   2024-12-15  2025-05-12
review ids shared: 0 | reviewers in both: 0
dates with the same availability: 365 of 365
free days: {1016487917711700538: 356, 1035296111642091276: 356}


Their shared calendar may belong to the host and not to one home. Compare the other listings of this host with the same name, by counting how many have the same calendar as the first of the two.

In [122]:
host_name_rows = listings_clean[(listings_clean["host_id"] == closest["host_id"].iloc[0]) & (listings_clean["name"] == closest["name"].iloc[0])]
host_calendar = calendar_clean[calendar_clean["listing_id"].isin(host_name_rows["id"])].pivot(index="date", columns="listing_id", values="available")
print("listings:", len(host_name_rows), "| distinct coordinates:", host_name_rows[["latitude", "longitude"]].drop_duplicates().shape[0])
print("free days:", host_name_rows["availability_365"].value_counts().to_dict())
print("same calendar as the first:", host_calendar.eq(host_calendar[closest_ids[0]], axis=0).all().sum(), "of", host_calendar.shape[1])

listings: 11 | distinct coordinates: 4
free days: {356: 6, 357: 2, 264: 1, 351: 1, 354: 1}
same calendar as the first: 6 of 11


**Findings:** the hypothesis fits what was checked: no pair is shown to be one home listed twice, although that cannot be ruled out completely.
- **Same host and coordinates:** 804 listings in 226 groups, for example `Ursa Minor` and `Ursa Major`. Their names and prices differ, and they have separate reviews.
- **Same host and name:** 108 listings in 32 groups. Many are at different coordinates, like the 3 `F & B Collection - 2 Bedroom Flat`, so the name is a type of flat that the host repeats.
- **Both:** 44 listings. 38 of them also match on room type, guests, bedrooms and price, but only 2 also have the same number of reviews.
- **The 2 closest** (`DELUXE ΔΙΑΜΕΡΙΣΜΑ`) share no review and no reviewer, and their first reviews are from different dates (2024-02-03 and 2024-12-15). So they have separate histories. Their availability is the same on all 365 dates, with 356 free days each, but this is not specific to them: 6 of the 11 listings of this host with this name have the same calendar, and the 11 are at 4 different coordinates. So the calendar seems to come from how the host manages these listings.
- **What the coordinates mean:** they can be up to 150 metres from the real address, so "same coordinates" does not prove the same building, and different coordinates do not prove different buildings. The page also says that listings in the same building are anonymized individually, which does not explain why 804 listings share exact coordinates with another listing of their host. The data does not show the reason. So coordinates alone cannot be used to detect duplicates.

**Decision:** recorded only. No listing is dropped: two flats in the same building are not an error.

### 6e. The host as one entity

**Hypothesis:** a host appears in many rows, so all the host columns have one value per `host_id`, and `calculated_host_listings_count` equals the real number of rows of each host.

For each host column, the number of hosts with more than one value. `dropna=False` counts NaN as a value, so a host with both NaN and text is also caught.

In [123]:
host_cols = [col for col in listings_clean.columns
             if col.startswith(("host_", "hosts_", "calculated_host")) and col != "host_id"]
print(len(host_cols), "columns,", listings_clean["host_id"].nunique(), "hosts")
(listings_clean.groupby("host_id")[host_cols].nunique(dropna=False) > 1).sum()

19 columns, 2122 hosts


host_url                                        0
host_profile_id                                 0
host_profile_url                                0
host_name                                       0
hosts_time_as_user_years                        0
hosts_time_as_user_months                       0
hosts_time_as_host_years                        0
hosts_time_as_host_months                       0
host_location                                   0
host_about                                      0
host_is_superhost                               0
host_picture_url                                0
host_listings_count                             0
host_has_profile_pic                            0
host_identity_verified                          0
calculated_host_listings_count                  0
calculated_host_listings_count_entire_homes     0
calculated_host_listings_count_private_rooms    0
calculated_host_listings_count_shared_rooms     0
dtype: int64

`calculated_host_listings_count` against the real number of rows of each host, and against the sum of its three room-type counts.

In [124]:
host_rows = listings_clean["host_id"].map(listings_clean["host_id"].value_counts())
print("equal to the real count:", (listings_clean["calculated_host_listings_count"] == host_rows).sum(), "of", len(listings_clean))
type_cols = ["calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
             "calculated_host_listings_count_shared_rooms"]
print("equal to the sum by room type:", (listings_clean[type_cols].sum(axis=1) == listings_clean["calculated_host_listings_count"]).sum())

equal to the real count: 4961 of 4961
equal to the sum by room type: 4961


`host_listings_count` against the same real count. The data dictionary says it comes from "Airbnb unknown calculations".

In [125]:
print("equal:", (listings_clean["host_listings_count"] == host_rows).sum(),
      "| higher:", (listings_clean["host_listings_count"] > host_rows).sum(),
      "| lower:", (listings_clean["host_listings_count"] < host_rows).sum())

equal: 2765 | higher: 2196 | lower: 0


**Findings:** the hypothesis holds.
- All 19 host columns have one value per host, for all 2,122 hosts, counting NaN as a value.
- `calculated_host_listings_count` equals the real number of rows of the host in all 4,961 rows, and also the sum of its three room-type counts.
- `host_listings_count` is a different number: it is equal in 2,765 rows, higher in 2,196 and never lower. The dictionary calls it Airbnb's own calculation and says that `calculated_host_listings_count` covers this city or region, so it may also count listings elsewhere. The data cannot confirm that.

**Decision:** nothing to fix. The EDA should use `calculated_host_listings_count` for the number of listings a host has in this file.

### 6f. Referential integrity

**Hypothesis:** every row of `calendar` and `reviews` points to a listing that exists, every listing is in `calendar`, and the listings with no rows in `reviews` are the same 571 as in 5a.

Orphans are rows whose `listing_id` is not in `listings` (`~` swaps True and False). Coverage counts the listings that do not appear in the table.

In [126]:
for name in ["calendar", "reviews"]:
    df = dataframes_clean[name]
    print(name, "| orphan rows:", (~df["listing_id"].isin(listings_clean["id"])).sum(),
          "| listings not in the table:", (~listings_clean["id"].isin(df["listing_id"])).sum())

calendar | orphan rows: 0 | listings not in the table: 0
reviews | orphan rows: 0 | listings not in the table: 571


The fewest and the most rows per listing, to confirm that the relationship is N:1.

In [127]:
for name in ["calendar", "reviews"]:
    rows_per_listing = dataframes_clean[name].groupby("listing_id").size()
    print(name, "| fewest:", rows_per_listing.min(), "| most:", rows_per_listing.max())

calendar | fewest: 365 | most: 365
reviews | fewest: 1 | most: 955


Compare the two groups as sets of ids, not only as counts: the listings not in `reviews`, and the listings with no review values in 5a.

In [128]:
not_in_reviews = set(listings_clean.loc[~listings_clean["id"].isin(reviews_clean["listing_id"]), "id"])
no_values_5a = set(listings_clean.loc[no_review_values, "id"])
print(len(not_in_reviews), len(no_values_5a), "| same ids:", not_in_reviews == no_values_5a)

571 571 | same ids: True


**Findings:** the hypothesis holds.

| relationship | type | orphan rows | listings not covered | rows per listing |
|---|---|---|---|---|
| `calendar.listing_id` → `listings.id` | N:1 | 0 | 0 | 365 for all |
| `reviews.listing_id` → `listings.id` | N:1 | 0 | 571 | 1 to 955 |

The 571 listings not in `reviews` are exactly the same ids as the 571 of 5a, compared as sets and not only as counts.

**Decision:** nothing to fix. A listing with no rows in `reviews` is a listing with no reviews, not a broken link.

### 6g. Summary

**Hypothesis:** this section changed no table: shapes, NaN and id dtypes are the same as at the end of Section 5.

The shapes and NaN of the three tables against Section 5, and the raw tables against a fresh read of their CSV.

In [129]:
for name, df in dataframes_clean.items():
    print(name, df.shape, "| NaN now:", df.isna().sum().sum(), "| in Section 5:", nan_start[name],
          "| raw unchanged:", dataframes[name].equals(pd.read_csv(f"data/{name}.csv")))

listings (4961, 78) | NaN now: 17263 | in Section 5: 17263 | raw unchanged: True
calendar (1810765, 5) | NaN now: 0 | in Section 5: 0 | raw unchanged: True
reviews (244369, 6) | NaN now: 14 | in Section 5: 14 | raw unchanged: True


The id columns of the `_clean` tables, and `dataframes_clean` against the latest tables.

In [130]:
for name, df in dataframes_clean.items():
    print(name, df[df.columns.intersection(id_cols)].dtypes.astype(str).to_dict())
print(dataframes_clean["listings"] is listings_clean,
      dataframes_clean["calendar"] is calendar_clean,
      dataframes_clean["reviews"] is reviews_clean)

listings {'id': 'int64', 'host_id': 'int64', 'host_profile_id': 'int64'}
calendar {'listing_id': 'int64'}
reviews {'listing_id': 'int64', 'id': 'int64', 'reviewer_id': 'int64'}
True True True


**Findings:** the hypothesis holds. The shapes are (4961, 78), (1810765, 5) and (244369, 6), the NaN are still 17,263, 0 and 14, the raw tables equal a fresh read of their CSV, all 7 id columns are `int64`, and `dataframes_clean` points to the latest tables.

The grain table of Section 2, now updated there too:

| table | one row is | key | status | found in this section |
|---|---|---|---|---|
| `listings` | one Airbnb listing | `id` | verified | no duplicate keys; no pair shown to be one home listed twice |
| `calendar` | one listing on one date | (`listing_id`, `date`) | verified | 365 rows per listing; 21 listings start a day earlier |
| `reviews` | one guest review | `id` | verified | no duplicate keys; 2 same-day groups and 23 long copies recorded |

This section dropped 0 rows and 0 columns, and filled 0 values.

**Goes to Section 7**, because it is the same number in two tables:
- `number_of_reviews` against the rows of each listing in `reviews`
- `first_review` and `last_review` against the dates in `reviews`
- `availability_365` against `calendar`, where 5b found 3 listings that disagree
- `minimum_nights` and `maximum_nights` against `calendar`